# AHS Farmer Data Cleaning + k-NN Pattern-Matching Model (v3)

`s3_q4_1` (crop name) is confirmed to be **text**, not a numeric AHS code —
so matching against the app's free-text `crop_type` field is a text-matching
problem, not a code-lookup problem. This still needs light normalization
(case, whitespace, singular/plural) since the app's spelling may not exactly
match AHS's convention (e.g. app: "Beans", AHS: "Bean").

Builds two k-NN indices:
- `knn_full` — district, crop name, season, sowing day
- `knn_no_crop` — district, season, sowing day only (used when the incoming
  crop name doesn't match any crop name seen in training, even after
  normalization/aliasing)

## 1. Load raw data

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import NearestNeighbors

In [2]:
import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
print("This version gets saved into the model bundle -- api.py checks it "
      "matches at load time. Make sure your environment matches "
      "requirements.txt before running this notebook.")

scikit-learn version: 1.6.1
This version gets saved into the model bundle -- api.py checks it matches at load time. Make sure your environment matches requirements.txt before running this notebook.


In [ ]:
raw = pd.read_csv(csv_path)
print(f"Loaded raw data from {csv_path} with shape {raw.shape}")
raw.head()

## 2. Extract the relevant columns

In [ ]:
columns_to_extract = [
    'hhid', 'district', 'Season', 's3_q4_1'
    , 's4_q1', 's4_q4', 's4_q12', 's4_q25_1', 's3_q20_1'
]

existing_columns = [c for c in columns_to_extract if c in raw.columns]
data = raw[existing_columns].copy()

print(f"Extracted {len(existing_columns)} columns.")
data.head()

Extracted 9 columns.


,hhid,district,Season,s3_q4_1,s4_q1,s4_q4,s4_q12,s4_q25_1,s3_q20_1
0,300307.0,11.0,2.0,106,1,2,2,NaN,30.0
1,300307.0,11.0,1.0,101,1,2,2,NaN,50.0
2,300307.0,11.0,2.0,106,1,2,2,NaN,20.0
3,300307.0,11.0,2.0,101,1,2,2,NaN,2.0
4,300307.0,11.0,3.0,106,2,2,2,NaN,30.0


## 3. Clean the data

- Normalize crop name text (strip whitespace, lowercase) so matching against
  the app's `crop_type` field is consistent.
- Drop rows with a missing outcome (`s3_q20_1`) rather than imputing it.
- Cap extreme harvest-quantity outliers at the 99th percentile.

In [ ]:
data = data.drop_duplicates()

data['s3_q4_1'] = data['s3_q4_1'].astype(str).str.strip().str.lower()

before = len(data)
data = data.dropna(subset=['s3_q20_1'])
print(f"Dropped {before - len(data)} rows with missing harvest quantity.")

cap = data['s3_q20_1'].quantile(0.99)
n_capped = (data['s3_q20_1'] > cap).sum()
data['s3_q20_1'] = data['s3_q20_1'].clip(upper=cap)
print(f"Capped {n_capped} outlier rows above {cap:.1f} kg.")

data['irrigated'] = data['s4_q25_1'].notna().astype(int)

print(f"Final cleaned row count: {len(data)}")
print("\nUnique crop names in the data (check these against the app's crop_type spellings):")
print(sorted(data['s3_q4_1'].unique()))

Dropped 2711 rows with missing harvest quantity.
Capped 288 outlier rows above 1750.0 kg.
Final cleaned row count: 28851

Unique crop names in the data (check these against the app's crop_type spellings):
['101', '102', '103', '104', '106', '107', '108', '109', '110', '111', '112', '113', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '126', '127', '128', '130', '131', '132', '134', '135', '136', '138', '139', '140', '141', '144', '201', '202', '203', '204', '205', '206', '301', '302', '303', '304', '305', '306', '309', '311', '401', '402', '403', '404', '406', '409', '410', '416', '501', '502', '507', '508', '510']


**Check the printed crop name list above against what the onboarding app
sends as `crop_type`.** Any spelling mismatch (e.g. "bean" vs "beans") should
be added to the `CROP_NAME_ALIASES` dict in `api.py` so it still routes to
the full match instead of falling back.

In [ ]:
data.shape

(28851, 10)

In [ ]:
crop_mapping = {
    101: "Maize",
    102: "Paddy rice",
    103: "Sorghum",
    104: "Wheat",
    105: "Other cereal (specify)",
    106: "Bush bean",
    107: "Climbing bean",
    108: "Pea",
    109: "Other pulse (specify)",
    110: "Irish potato",
    111: "Sweet potato",
    112: "Taro",
    113: "Yams",
    114: "Other tubers (specify)",
    115: "Tomato",
    116: "Cabbage",
    117: "Cauliflower",
    118: "Onion",
    119: "Carrot",
    120: "Eggplant",
    121: "Other seasonal vegetables (specify)",
    122: "Soybean",
    123: "Groundnut",
    124: "Sun flower",
    125: "Black eggplant",
    126: "Sweet pepper",
    127: "Amaranth",
    128: "Celery",
    129: "Spinach",
    130: "Small red bean",
    131: "Beet root",
    132: "Garlic",
    133: "African cabbage",
    134: "Leek",
    135: "French beans",
    136: "Letus",
    137: "Brocolli",
    138: "Millet",
    139: "Cucumber",
    140: "Chia seeds",
    141: "Other seasonal crops (specify)",
    142: "Levendine",
    143: "Rosemary",
    144: "Lemongrass",
    145: "Geranium",
    146: "Vetiver",
    147: "Patchouli",
    148: "Artemesia",
    149: "Mint",
    201: "Pyrethrum",
    202: "Pepper",
    203: "Pumpkin",
    204: "Napia grass",
    205: "Sugar cane",
    206: "Tobacco",
    207: "Other annual crops (specify)",
    301: "Cooking banana",
    302: "Dessert banana",
    303: "Banana for beer",
    304: "Coffee",
    305: "Cassava",
    306: "Mulberry",
    307: "Jatropha",
    308: "Stevia",
    309: "Macadamia",
    310: "Tea",
    311: "Other perennial crop (specify)",
    401: "Tree tomato",
    402: "Pineapple",
    403: "Avocado",
    404: "Passion fruits",
    405: "Palm",
    406: "Mango",
    407: "Apple",
    408: "Papaya",
    409: "Orange",
    410: "Lemon",
    411: "Guava",
    412: "Olive",
    413: "Water melon",
    414: "Mandoline",
    415: "Jack Fruits",
    416: "Goosebery",
    417: "Strawberry",
    418: "Coeur de boeuf",
    419: "Other fruits (specify)",
    501: "Napia grass for fodder",
    502: "Maize for fodder",
    503: "Soybean for fodder",
    504: "Leucena",
    505: "Desmodium",
    506: "Mucuna",
    507: "Setaria",
    508: "Tripsacum",
    509: "Herbaceous",
    510: "Other fodder crop (specify)"
}

# Convert dictionary keys to string to match the 's3_q4_1' column type
string_crop_mapping = {str(k): v for k, v in crop_mapping.items()}

data['s3_q4_1'] = data['s3_q4_1'].replace(string_crop_mapping)

print("Crop names replaced in 's3_q4_1' column.")
print("Updated unique crop names:")
print(sorted(data['s3_q4_1'].unique()))

Crop names replaced in 's3_q4_1' column.
Updated unique crop names:
['Amaranth', 'Avocado', 'Banana for beer', 'Beet root', 'Bush bean', 'Cabbage', 'Carrot', 'Cassava', 'Cauliflower', 'Celery', 'Chia seeds', 'Climbing bean', 'Coffee', 'Cooking banana', 'Cucumber', 'Dessert banana', 'Eggplant', 'French beans', 'Garlic', 'Goosebery', 'Groundnut', 'Irish potato', 'Leek', 'Lemon', 'Lemongrass', 'Letus', 'Macadamia', 'Maize', 'Maize for fodder', 'Mango', 'Millet', 'Mulberry', 'Napia grass', 'Napia grass for fodder', 'Onion', 'Orange', 'Other fodder crop (specify)', 'Other perennial crop (specify)', 'Other pulse (specify)', 'Other seasonal crops (specify)', 'Other seasonal vegetables (specify)', 'Paddy rice', 'Passion fruits', 'Pea', 'Pepper', 'Pineapple', 'Pumpkin', 'Pyrethrum', 'Setaria', 'Small red bean', 'Sorghum', 'Soybean', 'Sugar cane', 'Sun flower', 'Sweet pepper', 'Sweet potato', 'Taro', 'Tobacco', 'Tomato', 'Tree tomato', 'Tripsacum', 'Wheat', 'Yams']


In [ ]:
district_mapping = {
    11: "Nyarugenge",
    12: "Gasabo",
    13: "Kicukiro",
    21: "Nyanza",
    22: "Gisagara",
    23: "Nyaruguru",
    24: "Huye",
    25: "Nyamagabe",
    26: "Ruhango",
    27: "Muhanga",
    28: "Kamonyi",
    31: "Karongi",
    32: "Rutsiro",
    33: "Rubavu",
    34: "Nyabihu",
    35: "Ngororero",
    36: "Rusizi",
    37: "Nyamasheke",
    41: "Rulindo",
    42: "Gakenke",
    43: "Musanze",
    44: "Burera",
    45: "Gicumbi",
    51: "Rwamagana",
    52: "Nyagatare",
    53: "Gatsibo",
    54: "Kayonza",
    55: "Kirehe",
    56: "Ngoma",
    57: "Bugesera"
}

# Convert dictionary keys to float to match the 'district' column type, as seen in data.head()
float_district_mapping = {float(k): v for k, v in district_mapping.items()}

data['district'] = data['district'].replace(float_district_mapping)

print("District codes replaced in 'district' column.")
print("Updated unique district names:")
print(sorted(data['district'].unique()))

District codes replaced in 'district' column.
Updated unique district names:
['Bugesera', 'Burera', 'Gakenke', 'Gasabo', 'Gatsibo', 'Gicumbi', 'Gisagara', 'Huye', 'Kamonyi', 'Karongi', 'Kayonza', 'Kicukiro', 'Kirehe', 'Muhanga', 'Musanze', 'Ngoma', 'Ngororero', 'Nyabihu', 'Nyagatare', 'Nyamagabe', 'Nyamasheke', 'Nyanza', 'Nyarugenge', 'Nyaruguru', 'Rubavu', 'Ruhango', 'Rulindo', 'Rusizi', 'Rutsiro', 'Rwamagana']


In [ ]:
data.head()

,hhid,district,Season,s3_q4_1,s4_q1,s4_q4,s4_q12,s4_q25_1,s3_q20_1,irrigated
0,300307.0,Nyarugenge,2.0,Bush bean,1,2,2,NaN,30.0,0
1,300307.0,Nyarugenge,1.0,Maize,1,2,2,NaN,50.0,0
2,300307.0,Nyarugenge,2.0,Bush bean,1,2,2,NaN,20.0,0
3,300307.0,Nyarugenge,2.0,Maize,1,2,2,NaN,2.0,0
4,300307.0,Nyarugenge,3.0,Bush bean,2,2,2,NaN,30.0,0


In [ ]:
pesticide_mapping = {
    1: 'yes',
    2: 'no'
}

# Assuming s4_q12 might be float or int, apply mapping
data['s4_q12'] = data['s4_q12'].replace(pesticide_mapping)

# Rename the column
data = data.rename(columns={'s4_q12': 'Pesticide use'})

print("Values in 'Pesticide use' column updated and column renamed.")
display(data.head())

Values in 'Pesticide use' column updated and column renamed.


,hhid,district,Season,s3_q4_1,s4_q1,s4_q4,Pesticide use,s4_q25_1,s3_q20_1,irrigated
0,300307.0,Nyarugenge,2.0,Bush bean,1,2,no,NaN,30.0,0
1,300307.0,Nyarugenge,1.0,Maize,1,2,no,NaN,50.0,0
2,300307.0,Nyarugenge,2.0,Bush bean,1,2,no,NaN,20.0,0
3,300307.0,Nyarugenge,2.0,Maize,1,2,no,NaN,2.0,0
4,300307.0,Nyarugenge,3.0,Bush bean,2,2,no,NaN,30.0,0


In [ ]:
fertilizer_mapping = {
    1: 'yes',
    2: 'no'
}

# Apply mapping to s4_q1 (organic fertilizer)
data['s4_q1'] = data['s4_q1'].replace(fertilizer_mapping)

# Apply mapping to s4_q4 (inorganic fertilizer)
data['s4_q4'] = data['s4_q4'].replace(fertilizer_mapping)

display(data.head())

,hhid,district,Season,s3_q4_1,s4_q1,s4_q4,Pesticide use,s4_q25_1,s3_q20_1,irrigated
0,300307.0,Nyarugenge,2.0,Bush bean,yes,no,no,NaN,30.0,0
1,300307.0,Nyarugenge,1.0,Maize,yes,no,no,NaN,50.0,0
2,300307.0,Nyarugenge,2.0,Bush bean,yes,no,no,NaN,20.0,0
3,300307.0,Nyarugenge,2.0,Maize,yes,no,no,NaN,2.0,0
4,300307.0,Nyarugenge,3.0,Bush bean,no,no,no,NaN,30.0,0


In [ ]:
data = data.rename(columns={
    's4_q1': 'Organic fertilizer use',
    'Inorganic fertilizer used': 'Inorganic fertilizer use'
})

print("Columns 's4_q1' and 's4_q4' have been renamed.")
display(data.head())

Columns 's4_q1' and 's4_q4' have been renamed.


,hhid,district,Season,s3_q4_1,Organic fertilizer use,Inorganic fertilizer use,Pesticide use,s4_q25_1,s3_q20_1,irrigated
0,300307.0,Nyarugenge,2.0,Bush bean,yes,no,no,NaN,30.0,0
1,300307.0,Nyarugenge,1.0,Maize,yes,no,no,NaN,50.0,0
2,300307.0,Nyarugenge,2.0,Bush bean,yes,no,no,NaN,20.0,0
3,300307.0,Nyarugenge,2.0,Maize,yes,no,no,NaN,2.0,0
4,300307.0,Nyarugenge,3.0,Bush bean,no,no,no,NaN,30.0,0


In [ ]:
data = data.rename(columns={'s3_q4_1': 'crop_name'})
data = data.drop(columns=['s4_q25_1'])

print("Column 's3_q4_1' renamed to 'crop_name' and 's4_q25_1' dropped.")
display(data.head())

Column 's3_q4_1' renamed to 'crop_name' and 's4_q25_1' dropped.


,hhid,district,Season,crop_name,Organic fertilizer use,Inorganic fertilizer use,Pesticide use,s3_q20_1,irrigated
0,300307.0,Nyarugenge,2.0,Bush bean,yes,no,no,30.0,0
1,300307.0,Nyarugenge,1.0,Maize,yes,no,no,50.0,0
2,300307.0,Nyarugenge,2.0,Bush bean,yes,no,no,20.0,0
3,300307.0,Nyarugenge,2.0,Maize,yes,no,no,2.0,0
4,300307.0,Nyarugenge,3.0,Bush bean,no,no,no,30.0,0


In [ ]:
data = data.rename(columns={'s3_q20_1': 'crop_yield(kg)'})

print("Column 's3_q20_1' renamed to 'crop_yield'.")
display(data.head())

Column 's3_q20_1' renamed to 'crop_yield'.


,hhid,district,Season,crop_name,Organic fertilizer use,Inorganic fertilizer use,Pesticide use,crop_yield(kg),irrigated
0,300307.0,Nyarugenge,2.0,Bush bean,yes,no,no,30.0,0
1,300307.0,Nyarugenge,1.0,Maize,yes,no,no,50.0,0
2,300307.0,Nyarugenge,2.0,Bush bean,yes,no,no,20.0,0
3,300307.0,Nyarugenge,2.0,Maize,yes,no,no,2.0,0
4,300307.0,Nyarugenge,3.0,Bush bean,no,no,no,30.0,0


In [ ]:
print("--- Data Info ---")
data.info()

print("\n--- Descriptive Statistics ---")
display(data.describe(include='all'))

--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 28851 entries, 0 to 32614
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   hhid                      28851 non-null  float64
 1   district                  28851 non-null  object 
 2   Season                    28851 non-null  float64
 3   crop_name                 28851 non-null  object 
 4   Organic fertilizer use    28851 non-null  object 
 5   Inorganic fertilizer use  28851 non-null  object 
 6   Pesticide use             28851 non-null  object 
 7   crop_yield(kg)            28851 non-null  float64
 8   irrigated                 28851 non-null  int64  
dtypes: float64(3), int64(1), object(5)
memory usage: 2.2+ MB

--- Descriptive Statistics ---


,hhid,district,Season,crop_name,Organic fertilizer use,Inorganic fertilizer use,Pesticide use,crop_yield(kg),irrigated
count,28851.000000,28851,28851.000000,28851,28851,28851,28851,28851.000000,28851.000000
unique,NaN,30,NaN,63,2,2,2,NaN,NaN
top,NaN,Gicumbi,NaN,Maize,yes,no,no,NaN,NaN
freq,NaN,1645,NaN,6225,18885,20156,24237,NaN,NaN
mean,308149.007903,NaN,1.591626,NaN,NaN,NaN,NaN,128.392233,0.054106
std,3919.574602,NaN,0.582130,NaN,NaN,NaN,NaN,255.633588,0.226230
min,300307.000000,NaN,1.000000,NaN,NaN,NaN,NaN,0.000000,0.000000
25%,304647.000000,NaN,1.000000,NaN,NaN,NaN,NaN,15.000000,0.000000
50%,308390.000000,NaN,2.000000,NaN,NaN,NaN,NaN,40.000000,0.000000
75%,311379.000000,NaN,2.000000,NaN,NaN,NaN,NaN,120.000000,0.000000


In [ ]:
# Checking the 'irrigated' column derivation and values
print("Value counts for 'irrigated':")
print(data['irrigated'].value_counts())

print("\nSample of rows (Irrigated vs Not Irrigated):")
display(data[['irrigated', 'crop_name']].head(10))

Value counts for 'irrigated':
irrigated
0    27290
1     1561
Name: count, dtype: int64

Sample of rows (Irrigated vs Not Irrigated):


,irrigated,crop_name
0,0,Bush bean
1,0,Maize
2,0,Bush bean
3,0,Maize
4,0,Bush bean
5,0,Bush bean
7,0,Banana for beer
8,1,Bush bean
9,0,Bush bean
10,1,Tomato


In [ ]:
irrigated_mapping = {1: 'yes', 0: 'no'}

data['irrigated'] = data['irrigated'].replace(irrigated_mapping)

print("Value counts for 'irrigated' (updated):")
print(data['irrigated'].value_counts())
display(data[['irrigated', 'crop_name']].head(10))

Value counts for 'irrigated' (updated):
irrigated
no     27290
yes     1561
Name: count, dtype: int64


,irrigated,crop_name
0,no,Bush bean
1,no,Maize
2,no,Bush bean
3,no,Maize
4,no,Bush bean
5,no,Bush bean
7,no,Banana for beer
8,yes,Bush bean
9,no,Bush bean
10,yes,Tomato


In [ ]:
data.head(
)

,hhid,district,Season,crop_name,Organic fertilizer use,Inorganic fertilizer use,Pesticide use,crop_yield(kg),irrigated
0,300307.0,Nyarugenge,2.0,Bush bean,yes,no,no,30.0,no
1,300307.0,Nyarugenge,1.0,Maize,yes,no,no,50.0,no
2,300307.0,Nyarugenge,2.0,Bush bean,yes,no,no,20.0,no
3,300307.0,Nyarugenge,2.0,Maize,yes,no,no,2.0,no
4,300307.0,Nyarugenge,3.0,Bush bean,no,no,no,30.0,no


## 4. Build two k-NN matching models

In [ ]:
full_features = ['district', 'crop_name', 'Season', 'Pesticide use', 'Organic fertilizer use', 'Inorganic fertilizer use', 'irrigated']
no_crop_features = ['district', 'Season', 'Pesticide use', 'Organic fertilizer use', 'Inorganic fertilizer use', 'irrigated']

def build_preprocessor(cat_cols, num_cols):
    return ColumnTransformer(transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), num_cols)
    ])

# District, crop_name, Season, and the yes/no columns are all categorical now
cat_features_full = ['district', 'crop_name', 'Season', 'Pesticide use', 'Organic fertilizer use', 'Inorganic fertilizer use', 'irrigated']
preprocessor_full = build_preprocessor(cat_features_full, [])
X_full = preprocessor_full.fit_transform(data[full_features])
knn_full = NearestNeighbors(n_neighbors=15, metric='euclidean')
knn_full.fit(X_full)

cat_features_no_crop = ['district', 'Season', 'Pesticide use', 'Organic fertilizer use', 'Inorganic fertilizer use', 'irrigated']
preprocessor_no_crop = build_preprocessor(cat_features_no_crop, [])
X_no_crop = preprocessor_no_crop.fit_transform(data[no_crop_features])
knn_no_crop = NearestNeighbors(n_neighbors=15, metric='euclidean')
knn_no_crop.fit(X_no_crop)

known_crops = sorted(data['crop_name'].unique().tolist())

print("Both k-NN models fitted on", len(data), "records.")
print("Known crop names:", known_crops)

Both k-NN models fitted on 28851 records.
Known crop names: ['Amaranth', 'Avocado', 'Banana for beer', 'Beet root', 'Bush bean', 'Cabbage', 'Carrot', 'Cassava', 'Cauliflower', 'Celery', 'Chia seeds', 'Climbing bean', 'Coffee', 'Cooking banana', 'Cucumber', 'Dessert banana', 'Eggplant', 'French beans', 'Garlic', 'Goosebery', 'Groundnut', 'Irish potato', 'Leek', 'Lemon', 'Lemongrass', 'Letus', 'Macadamia', 'Maize', 'Maize for fodder', 'Mango', 'Millet', 'Mulberry', 'Napia grass', 'Napia grass for fodder', 'Onion', 'Orange', 'Other fodder crop (specify)', 'Other perennial crop (specify)', 'Other pulse (specify)', 'Other seasonal crops (specify)', 'Other seasonal vegetables (specify)', 'Paddy rice', 'Passion fruits', 'Pea', 'Pepper', 'Pineapple', 'Pumpkin', 'Pyrethrum', 'Setaria', 'Small red bean', 'Sorghum', 'Soybean', 'Sugar cane', 'Sun flower', 'Sweet pepper', 'Sweet potato', 'Taro', 'Tobacco', 'Tomato', 'Tree tomato', 'Tripsacum', 'Wheat', 'Yams']


## 5. Quick sanity check

In [ ]:
data.columns

Index(['hhid', 'district', 'Season', 'crop_name', 'Organic fertilizer use',
       'Inorganic fertilizer use', 'Pesticide use', 'crop_yield(kg)',
       'irrigated'],
      dtype='object')

In [ ]:
reference_df = data[['hhid', 'district', 'Season', 'crop_name', 'Organic fertilizer use',
       'Inorganic fertilizer use', 'Pesticide use', 'crop_yield(kg)',
       'irrigated']].reset_index(drop=True)

# The query must include all features used in 'full_features'
test_query_full = pd.DataFrame([{
    'district': data['district'].iloc[0],
    'crop_name': data['crop_name'].iloc[0],
    'Season': 1,
    'Pesticide use': 'no',
    'Organic fertilizer use': 'yes',
    'Inorganic fertilizer use': 'no',
    'irrigated': 'no'
}])

q_full = preprocessor_full.transform(test_query_full)
_, idx_full = knn_full.kneighbors(q_full)

print("Full-match neighbors:")
display(reference_df.iloc[idx_full[0]])

Full-match neighbors:


,hhid,district,Season,crop_name,Organic fertilizer use,Inorganic fertilizer use,Pesticide use,crop_yield(kg),irrigated
5,300307.0,Nyarugenge,1.0,Bush bean,yes,no,no,20.0,no
58,300634.0,Nyarugenge,1.0,Bush bean,yes,no,no,40.0,no
60,300634.0,Nyarugenge,1.0,Bush bean,yes,no,no,25.0,no
64,300635.0,Nyarugenge,1.0,Bush bean,yes,no,no,20.0,no
54,300630.0,Nyarugenge,1.0,Bush bean,yes,no,no,29.0,no
18,300309.0,Nyarugenge,1.0,Eggplant,yes,no,no,15.0,no
906,302221.0,Nyanza,1.0,Bush bean,yes,no,no,40.0,no
900,302220.0,Nyanza,1.0,Bush bean,yes,no,no,16.0,no
897,302219.0,Nyanza,1.0,Bush bean,yes,no,no,225.0,no
894,302219.0,Nyanza,1.0,Bush bean,yes,no,no,6.0,no


## 6. Save the model bundle

In [ ]:
bundle = {
    'sklearn_version': sklearn.__version__,
    'preprocessor_full': preprocessor_full,
    'knn_full': knn_full,
    'full_features': full_features,
    'preprocessor_no_crop': preprocessor_no_crop,
    'knn_no_crop': knn_no_crop,
    'no_crop_features': no_crop_features,
    'reference_df': reference_df,
    'known_crops': known_crops,
}

joblib.dump(bundle, 'knn_recommender.joblib')
print("Saved model bundle to knn_recommender.joblib")

Saved model bundle to knn_recommender.joblib
